# ex01 · 互相关运算与训练卷积核（对应教材 6.2 图像卷积）

> **做题流程**：从零实现 corr2d → 用它找边缘 → 再训练一个卷积核学出边缘检测器。
> **做完再看** `solutions/ex01-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）

## 题 1 🔧 从零实现二维互相关（TODO 6.1）

互相关 = 卷积核在输入上滑动，对应位置相乘再求和。先补全 corr2d。
（手算练习：X=[[0,1,2],[3,4,5],[6,7,8]]、K=[[0,1],[2,3]] 应得 [[19,25],[37,43]]）

In [ ]:
import torch
from torch import nn

def corr2d(X, K):
    # TODO 6.1: 输出形状 (X高-K高+1, X宽-K宽+1)，每个位置 = (X 的滑动窗口 * K).sum()
    raise NotImplementedError('⚠ TODO 6.1: corr2d 未完成')

In [ ]:
try:
    X = torch.tensor([[0.,1.,2.],[3.,4.,5.],[6.,7.,8.]])
    K = torch.tensor([[0.,1.],[2.,3.]])
    Y = corr2d(X, K)
    assert Y.tolist() == [[19.0, 25.0], [37.0, 43.0]], f'{Y.tolist()}'
    print('✓ corr2d 正确:', Y.tolist())
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

## 题 2 🔧 用卷积找图像边缘

一张图中间涂黑，用核 `K=[[1,-1]]`（水平差分）扫一遍，检测出「颜色变化」的竖直边缘。

In [ ]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
print('输入图像（1=白，0=黑）:\n', X)
print('\n边缘检测结果（非零处 = 颜色变化处）:\n', corr2d(X, torch.tensor([[1.0, -1.0]])))

## 题 3 🔧 从零训练一个卷积核（TODO 6.2）

卷积核是**可学习的参数**——不用手写 [[1,-1]]，而是让网络自己「学」出一个边缘检测器。用 nn.Conv2d 拟合题 2 的输入输出，补全训练循环。

In [ ]:
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)
X4 = X.reshape((1, 1, 6, 8))                            # 加 batch、通道维
Y4 = corr2d(X, torch.tensor([[1.0, -1.0]])).reshape((1, 1, 6, 7))

lr = 3e-2
for i in range(10):
    # TODO 6.2: 训练循环 —— Y_hat=conv2d(X4)；l=(Y_hat-Y4)**2；zero_grad；l.sum().backward()；手动更新 conv2d.weight
    raise NotImplementedError('⚠ TODO 6.2: 训练循环未完成')

print('学到的卷积核:', conv2d.weight.data.reshape(1, 2))

## 小结与面试衔接

- 互相关 = 卷积核滑动做逐元素乘加；卷积核是可学习参数，训练时自动学到特征提取器
- 不同核提取不同特征：[[1,-1]] 检测竖直边缘、[[-1],[1]] 检测水平边缘
- 从零训练卷积核 ≈ 深度学习的本质：不手工设计特征，让网络自己学